# AI Meeting Assistant — Stage 3 (Minutes, decisions, action items — LLM #2)

**Runtime → Change runtime type → T4 GPU.** Input is `stage2_record.json` from the Stage 2 notebook.

This stage is worth 40 of the 100 points (minutes & decisions 25, action items 15). The rubric wording is
"without invented claims" and "owners and deadlines only when stated", so the design is:

**the LLM proposes items with evidence quotes; code verifies every quote, owner and deadline against the
transcript before anything enters the record.** Proposals are classified and kept apart from decisions.
Empty lists are valid. The Markdown is rendered from the verified JSON by code, so both formats always agree.

| Step | Cell |
|---|---|
| Load refined transcript (Stage 3 never sees the raw one) | 4 |
| Numbered lines + chunking for long meetings | 5 |
| Documenter prompt (Appendix A) | 6 |
| Verifier: quote exists, owner/deadline in evidence, type filter | 7 |
| Self-test the verifier (no model) | 8 |
| Load Qwen2.5-7B-Instruct 4-bit, call + parse | 9–10 |
| Run, merge, final record | 11–12 |
| Render Markdown from JSON | 13 |
| Save `meeting_record.json` / `.md` + verification log | 14 |
| Evaluate against gold labels | 15 |

Declared documenter model: **Qwen/Qwen2.5-7B-Instruct (Apache-2.0), 4-bit NF4**. Distinct from the Stage 2 refiner.

## Cell 1 — Runtime check

In [ ]:
import subprocess, sys
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("GPU :", gpu or "NONE -> Runtime > Change runtime type > T4 GPU")
print("python:", sys.version.split()[0])

GPU : NONE -> Runtime > Change runtime type > T4 GPU
python: 3.13.16


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELS_DIR = Path("/content/drive/MyDrive/meeting-assistant/models")
except Exception as e:
    print("Drive not mounted ->", type(e).__name__, "| using local cache (lost on runtime reset)")
    MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Model cache:", MODELS_DIR)
for p in sorted(MODELS_DIR.glob("*")):
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
    print(f"  cached: {p.name:<40} {size:5.1f} GB")

## Cell 2 — Install

In [ ]:
!pip install -q "transformers>=4.45" accelerate bitsandbytes rapidfuzz
!pip install -q "transformers>=4.45" accelerate bitsandbytes rapidfuzz icalendar

import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## Cell 3 — Configuration (becomes the documenter section of `config.py`)

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class DocConfig:
    model_id: str       = "Qwen/Qwen2.5-7B-Instruct"   # distinct from the Stage 2 refiner
    load_in_4bit: bool  = True
    max_new_tokens: int = 1800
    # chunking (map-reduce) — a chunk is sent as one call; most demo clips fit in a single chunk
    max_chunk_words: int = 3500
    chunk_overlap_lines: int = 4
    # verifier
    quote_min_ratio: int = 85        # rapidfuzz partial_ratio for quote-in-evidence (tolerates punctuation)
    name_min_ratio: int  = 90        # owner must appear in evidence text (fuzzy, for case/punctuation)
    max_quote_words: int = 60
    meeting_context: str = "Technical team meeting (software / engineering)."

CFG = DocConfig()
WORK_DIR = Path("/content/meeting-assistant"); OUT_DIR = WORK_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG

## Cell 4 — Load Stage 2 output

Upload `stage2_record.json` (Stage 2 notebook, Cell 15). Stage 3 reads **only** `refined_transcript`.
Also accepts a plain `refined_transcript.txt` / `refined_transcript_numbered.txt`.

In [ ]:
import json, re
from google.colab import files

uploaded = files.upload()
STAGE2_PATH = str(WORK_DIR / next(iter(uploaded)))
Path(STAGE2_PATH).write_bytes(next(iter(uploaded.values())))
# STAGE2_PATH = "/content/drive/MyDrive/meeting-assistant/outputs/<run>/stage2_record.json"

def _segments_from_text(text: str) -> list[dict]:
    numbered = re.findall(r"\[L(\d+)\s+([\d:\-]+)\]\s*(.*)", text)
    if numbered:                                                       # refined_transcript_numbered.txt
        return [{"id": int(i), "start": None, "end": None, "text": t.strip()} for i, _, t in numbered]
    sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", text.strip()) if t.strip()]
    return [{"id": i + 1, "start": None, "end": None, "text": t} for i, t in enumerate(sents)]

def load_stage2(path: str) -> dict:
    p = Path(path); raw = p.read_text(encoding="utf-8")
    if p.suffix.lower() == ".txt":
        return {"segments": _segments_from_text(raw), "record": {"refined_transcript": {"text": raw}}, "meta": {}}
    d = json.loads(raw)
    rt = d.get("refined_transcript")
    if isinstance(rt, dict) and rt.get("segments"):
        segs = rt["segments"]
    elif isinstance(rt, dict) and rt.get("text"):
        segs = _segments_from_text(rt["text"])
    elif isinstance(rt, str):
        segs = _segments_from_text(rt)
    else:
        raise ValueError(f"No refined_transcript found. Keys: {list(d.keys())}. Run Stage 2 first.")
    segs = [{"id": s.get("id", i + 1), "start": s.get("start"), "end": s.get("end"), "text": str(s["text"]).strip()}
            for i, s in enumerate(segs) if str(s.get("text", "")).strip()]
    return {"segments": segs, "record": d, "meta": d.get("meta", {})}

S2 = load_stage2(STAGE2_PATH)
SEGMENTS = S2["segments"]
print(f"{len(SEGMENTS)} segments | {sum(len(s['text'].split()) for s in SEGMENTS)} words | models so far: "
      f"{S2['meta'].get('models')}")

## Cell 5 — Numbered lines and chunking

Every line gets `[L012 03:41]`. The model must cite these IDs as evidence. Long meetings are split into
word-budgeted chunks with a few lines of overlap (map step); Cell 11 merges them (reduce step).

In [ ]:
def fmt_ts(sec) -> str:
    if sec is None: return "--:--"
    sec = int(sec); return f"{sec//60:02d}:{sec%60:02d}"

LINES = {s["id"]: s["text"] for s in SEGMENTS}
def numbered(segments: list[dict]) -> str:
    return "\n".join(f"[L{s['id']:03d} {fmt_ts(s['start'])}] {s['text']}" for s in segments)

def make_chunks(segments: list[dict], max_words: int, overlap: int) -> list[list[dict]]:
    chunks, cur, words = [], [], 0
    for s in segments:
        w = len(s["text"].split())
        if cur and words + w > max_words:
            chunks.append(cur); cur = cur[-overlap:] if overlap else []; words = sum(len(x["text"].split()) for x in cur)
        cur.append(s); words += w
    if cur: chunks.append(cur)
    return chunks

CHUNKS = make_chunks(SEGMENTS, CFG.max_chunk_words, CFG.chunk_overlap_lines)
print(f"{len(CHUNKS)} chunk(s)\n" + numbered(SEGMENTS)[:800])

## Cell 6 — Prompts (copy to `prompts/document_system.txt`)

Appendix A documenter prompt, with the candidate-classification scheme and explicit examples of what is and
is not a decision.

In [ ]:
SYSTEM_PROMPT = '''You write meeting minutes from a refined meeting transcript. Each line is numbered like [L012 03:41].
Use ONLY what was said. Never infer, never guess, never add recommendations of your own.

Return ONLY a JSON object with exactly these keys:
{
 "summary": "3-5 sentence overview of the meeting",
 "discussion_points": [{"topic": "short topic name", "points": ["concise point", ...]}],
 "candidates": [{"statement": "what was said, concisely",
                 "type": "agreed_decision" | "proposal" | "open_question" | "tentative",
                 "evidence_quote": "verbatim words from the transcript",
                 "evidence_lines": [12, 13]}],
 "action_items": [{"task": "the work to be done",
                   "owner": "name exactly as spoken, or unspecified",
                   "owner_stated": true | false,
                   "deadline": "time exactly as spoken (e.g. by Friday), or unspecified",
                   "deadline_stated": true | false,
                   "evidence_quote": "verbatim words from the transcript",
                   "evidence_lines": [31]}]
}

Rules for "type":
- agreed_decision: the group clearly agreed or finalised it. "Okay, we're going with Postgres." / "We decided the release date is the 15th." / "Agreed."
- proposal: someone suggested it and it was NOT clearly accepted. "I think we could use Redis." / "Rahul suggested X, but nobody agreed."
- open_question: raised but not answered. "Should we move to AWS?"
- tentative: hedged. "Let's maybe revisit that." / "We should probably..."
A decision that was later reversed in the meeting must appear with its FINAL state only.

Rules for action items:
- Only commitments or assignments that were accepted or stated as a plan. "I'll send the report." / "Priya, can you handle it?" + "Sure".
- A suggestion with no acceptance is NOT a task. "Someone needs to write the runbook" IS a task, with owner unspecified.
- owner: copy the name exactly as spoken. If no name is spoken for that task, write "unspecified" and owner_stated=false. A speaker label is not a name.
- deadline: copy the time words exactly as spoken ("by Friday", "end of next week"). Never convert to a calendar date. If none spoken, "unspecified" and deadline_stated=false.
- Every candidate and action item MUST include evidence_quote (exact words from the lines) and evidence_lines.
- If there are no decisions or no action items, return empty lists. That is a correct answer.
- Be concise. No filler. No commentary outside the JSON.'''

def build_user_message(chunk: list[dict], context: str, part: str = "") -> str:
    return (f"Meeting context: {context}\n{part}"
            f"Transcript:\n{numbered(chunk)}\n\nReturn ONLY the JSON object.")

print(build_user_message(CHUNKS[0][:4], CFG.meeting_context)[:600])

## Cell 7 — The verifier (code decides what enters the record)

For each item: the evidence lines must exist, the quote must be found in those lines, decisions must be typed
`agreed_decision`, owners/deadlines must literally appear in the evidence text or they become `unspecified`.

In [ ]:
from rapidfuzz import fuzz

NEG_ACCEPT = ("nobody agreed", "no one agreed", "not agreed", "wasn't agreed", "was rejected", "we rejected",
              "decided against", "didn't agree", "did not agree", "not adopted", "shot down", "parked", "table that")

def _norm(s: str) -> str:
    return " ".join(re.sub(r"[^a-z0-9' ]", " ", str(s).lower()).split())

def evidence_text(lines_ids, lines: dict) -> str:
    ids = [int(i) for i in (lines_ids or []) if str(i).lstrip("L").isdigit() or isinstance(i, int)]
    return " ".join(lines[i] for i in ids if i in lines)

NEGATION_WORDS = {"not", "no", "never", "none", "nothing", "neither", "nor", "cannot", "without"}
def _negs(s: str) -> set[str]:
    return {w for w in _norm(s).split() if w in NEGATION_WORDS or w.endswith("n't")}

def quote_found(quote: str, ev_text: str, cfg: DocConfig) -> tuple[bool, int]:
    "Quote must occur in the evidence text: exact after normalisation, or fuzzy-aligned with identical negations."
    q, e = _norm(quote), _norm(ev_text)
    if not q or not e or len(q.split()) > cfg.max_quote_words:
        return False, 0
    if q in e:
        return True, 100
    al = fuzz.partial_ratio_alignment(q, e)
    r = int(al.score) if al else 0
    if r < cfg.quote_min_ratio:
        return False, r
    pad = 24                                         # ~3 tokens either side of the aligned span
    window = e[max(0, al.dest_start - pad): al.dest_end + pad]
    if _negs(q) != _negs(window):                    # "will migrate" vs "will not migrate" -> reject
        return False, -r
    return True, r

def _clean_ids(ids) -> list[int]:
    out = []
    for i in (ids or []):
        s = str(i).strip().lstrip("L")
        if s.isdigit(): out.append(int(s))
    return sorted(set(out))

def verify_candidate(c: dict, lines: dict, cfg: DocConfig) -> tuple[dict | None, str]:
    ids = _clean_ids(c.get("evidence_lines"))
    ev = evidence_text(ids, lines)
    if not ids or not ev:
        return None, "evidence lines missing/unknown"
    ok, score = quote_found(c.get("evidence_quote", ""), ev, cfg)
    if not ok:
        return None, ("quote changes a negation vs the evidence" if score < 0 else f"quote not found in evidence (ratio {score})")
    t = str(c.get("type", "")).strip().lower()
    if t not in ("agreed_decision", "proposal", "open_question", "tentative"):
        return None, f"unknown type {t!r}"
    # safety net: a "decision" whose own quoted words say it was not agreed is a proposal
    if t == "agreed_decision" and any(p in _norm(c.get("evidence_quote", "")) for p in NEG_ACCEPT):
        t = "proposal"
    return {"statement": str(c.get("statement", "")).strip(), "type": t,
            "evidence_quote": str(c.get("evidence_quote", "")).strip(), "evidence_lines": ids,
            "quote_match": score}, "ok"

def _name_in(name: str, ev: str, cfg: DocConfig) -> bool:
    n = _norm(name)
    if not n or n == "unspecified": return False
    toks = _norm(ev).split()
    return any(fuzz.ratio(n, " ".join(toks[i:i+len(n.split())])) >= cfg.name_min_ratio
               for i in range(len(toks)))

def verify_action(a: dict, lines: dict, cfg: DocConfig) -> tuple[dict | None, str, list[str]]:
    notes = []
    ids = _clean_ids(a.get("evidence_lines")); ev = evidence_text(ids, lines)
    if not ids or not ev:
        return None, "evidence lines missing/unknown", notes
    ok, score = quote_found(a.get("evidence_quote", ""), ev, cfg)
    if not ok:
        return None, ("quote changes a negation vs the evidence" if score < 0 else f"quote not found in evidence (ratio {score})"), notes
    task = str(a.get("task", "")).strip()
    if not task:
        return None, "empty task", notes
    owner = str(a.get("owner", "unspecified")).strip() or "unspecified"
    if owner.lower() != "unspecified" and not _name_in(owner, ev, cfg):
        notes.append(f"owner '{owner}' not in evidence -> unspecified"); owner = "unspecified"
    deadline = str(a.get("deadline", "unspecified")).strip() or "unspecified"
    if deadline.lower() != "unspecified":
        d_ok, _ = quote_found(deadline, ev, DocConfig(quote_min_ratio=80, max_quote_words=12))
        if not d_ok:
            notes.append(f"deadline '{deadline}' not in evidence -> unspecified"); deadline = "unspecified"
    if owner.lower() == "unspecified": owner = "unspecified"
    if deadline.lower() == "unspecified": deadline = "unspecified"
    return {"task": task, "owner": owner, "owner_stated": owner != "unspecified",
            "deadline": deadline, "deadline_stated": deadline != "unspecified",
            "evidence_quote": str(a.get("evidence_quote", "")).strip(), "evidence_lines": ids,
            "quote_match": score}, "ok", notes

def verify_output(obj: dict, lines: dict, cfg: DocConfig) -> dict:
    log = {"dropped": [], "adjusted": []}
    cands, actions = [], []
    for c in obj.get("candidates", []) or []:
        v, why = verify_candidate(c, lines, cfg)
        (cands.append(v) if v else log["dropped"].append({"kind": "candidate", "item": c, "reason": why}))
    for a in obj.get("action_items", []) or []:
        v, why, notes = verify_action(a, lines, cfg)
        if v:
            actions.append(v); log["adjusted"] += [{"task": v["task"], "note": n} for n in notes]
        else:
            log["dropped"].append({"kind": "action_item", "item": a, "reason": why})
    return {"summary": str(obj.get("summary", "")).strip(),
            "discussion_points": [{"topic": str(d.get("topic", "")).strip(), "points": [str(p) for p in d.get("points", [])]}
                                  for d in (obj.get("discussion_points") or []) if isinstance(d, dict)],
            "decisions": [c for c in cands if c["type"] == "agreed_decision"],
            "proposals_not_adopted": [c for c in cands if c["type"] in ("proposal", "tentative")],
            "open_questions": [c for c in cands if c["type"] == "open_question"],
            "action_items": actions, "verification": log}

print("verifier ready")

## Cell 8 — Self-test the verifier (no model needed)

Uses the playbook's test meeting. Covers: a real decision, a rejected proposal mislabelled as a decision,
a hallucinated quote, an invented owner, an invented deadline, and a task with neither.

In [ ]:
_lines = {1: "Okay everyone, let's start the weekly sync.",
          2: "First, the migration from MySQL to PostgreSQL.",
          3: "Priya says the staging database is ready, but we will not migrate production this quarter,",
          4: "because the load tests failed twice.",
          7: "Rahul suggested moving the API gateway to AWS, but nobody agreed, so we are keeping the current setup.",
          8: "Third, the launch. We decided the release date is the twenty third of October.",
          9: "Priya will update the Terraform scripts by Friday.",
          10: "Someone needs to write the rollback runbook.",
          11: "Rahul will send the benchmark report. That's all, thanks everyone."}
_out = {"summary": "s", "discussion_points": [],
 "candidates": [
  {"statement": "Release date is 23 October", "type": "agreed_decision", "evidence_quote": "We decided the release date is the twenty third of October", "evidence_lines": [8]},
  {"statement": "Move API gateway to AWS", "type": "agreed_decision", "evidence_quote": "Rahul suggested moving the API gateway to AWS, but nobody agreed", "evidence_lines": [7]},   # -> proposal
  {"statement": "No production migration this quarter", "type": "agreed_decision", "evidence_quote": "we will not migrate production this quarter", "evidence_lines": [3]},
  {"statement": "Adopt Redis caching", "type": "agreed_decision", "evidence_quote": "we agreed to adopt Redis for caching", "evidence_lines": [4]},   # hallucinated
  {"statement": "Keep the current gateway setup", "type": "agreed_decision", "evidence_quote": "so we are keeping the current setup", "evidence_lines": [7]},
  {"statement": "Migrate production this quarter", "type": "agreed_decision", "evidence_quote": "we will migrate production this quarter", "evidence_lines": [3]},  # negation flipped
 ],
 "action_items": [
  {"task": "Update Terraform scripts", "owner": "Priya", "owner_stated": True, "deadline": "by Friday", "deadline_stated": True, "evidence_quote": "Priya will update the Terraform scripts by Friday", "evidence_lines": [9]},
  {"task": "Write rollback runbook", "owner": "Rahul", "owner_stated": True, "deadline": "next week", "deadline_stated": True, "evidence_quote": "Someone needs to write the rollback runbook", "evidence_lines": [10]},  # invented owner+deadline
  {"task": "Send benchmark report", "owner": "Rahul", "owner_stated": True, "deadline": "unspecified", "deadline_stated": False, "evidence_quote": "Rahul will send the benchmark report", "evidence_lines": [11]},
  {"task": "Migrate production", "owner": "Priya", "owner_stated": True, "deadline": "this quarter", "deadline_stated": True, "evidence_quote": "Priya will migrate production this quarter", "evidence_lines": [3]},  # quote not found
 ]}
_v = verify_output(_out, _lines, CFG)
checks = [
  ("decision kept",            any("twenty third" in d["evidence_quote"] for d in _v["decisions"])),
  ("not-migrate kept",         any("will not migrate" in d["evidence_quote"] for d in _v["decisions"])),
  ("AWS -> proposal",          any("AWS" in p["statement"] for p in _v["proposals_not_adopted"]) and not any("AWS" in d["statement"] for d in _v["decisions"])),
  ("Redis dropped",            not any("Redis" in d["statement"] for d in _v["decisions"]) and any("Redis" in str(x["item"]) for x in _v["verification"]["dropped"])),
  ("Priya/Friday kept",        any(a["owner"] == "Priya" and a["deadline"] == "by Friday" for a in _v["action_items"])),
  ("runbook owner unspecified",any(a["task"].lower().startswith("write") and a["owner"] == "unspecified" and a["deadline"] == "unspecified" for a in _v["action_items"])),
  ("Rahul no deadline",        any(a["owner"] == "Rahul" and a["deadline"] == "unspecified" for a in _v["action_items"])),
  ("fake migrate task dropped",not any(a["task"] == "Migrate production" for a in _v["action_items"])),
  ("keep-setup kept as decision", any("current setup" in d["evidence_quote"] for d in _v["decisions"])),
  ("negation-flip dropped",    not any("will migrate production" in d["evidence_quote"] for d in _v["decisions"] + _v["proposals_not_adopted"])
                               and any("negation" in x["reason"] for x in _v["verification"]["dropped"])),
]
bad = 0
for name, ok in checks:
    print(("PASS " if ok else "FAIL ") + name); bad += not ok
print("\nALL GOOD" if not bad else f"\n{bad} verifier test(s) failed")

## Cell 9 — Load the documenter LLM (Qwen2.5-7B-Instruct, 4-bit)

Different model from Stage 2. If the Stage 2 model is still in this runtime, run its `free_llm()` first.

In [ ]:
import torch, time, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SAVE_4BIT_TO_DRIVE = True

def load_llm(cfg):
    t0 = time.time()
    local = MODELS_DIR / (cfg.model_id.replace("/", "__") + "-4bit")
    if (local / "config.json").exists():
        print("Loading 4-bit copy from Drive:", local)
        tok = AutoTokenizer.from_pretrained(local)
        model = AutoModelForCausalLM.from_pretrained(local, device_map="auto")
        source = "drive"
    else:
        print("Downloading from Hugging Face:", cfg.model_id, "(one-time)")
        tok = AutoTokenizer.from_pretrained(cfg.model_id)
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True) if cfg.load_in_4bit else None
        model = AutoModelForCausalLM.from_pretrained(cfg.model_id, quantization_config=quant,
                                                     device_map="auto", torch_dtype=torch.float16)
        source = "hub"
    model.eval()
    if source == "hub" and SAVE_4BIT_TO_DRIVE and cfg.load_in_4bit:
        print("Saving 4-bit copy to Drive (one-time, ~5.5 GB, a few minutes)...")
        model.save_pretrained(local, safe_serialization=True); tok.save_pretrained(local)
        print("Saved ->", local)
    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"Loaded {cfg.model_id} from {source} in {time.time()-t0:.0f}s | GPU mem {mem:.1f} GB")
    return tok, model

tokenizer, llm = load_llm(CFG)

def llm_generate(system: str, user: str, cfg: DocConfig = CFG) -> str:
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kw = {"enable_thinking": False} if "qwen3" in cfg.model_id.lower() else {}
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=cfg.max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

## Cell 10 — Call + parse (retry once; fall back to an empty, flagged result)

In [ ]:
EMPTY = {"summary": "", "discussion_points": [], "candidates": [], "action_items": []}

def parse_json_obj(text: str) -> dict | None:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m: return None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None

def document_chunk(chunk: list[dict], cfg: DocConfig, part: str = "") -> tuple[dict, str, dict]:
    user = build_user_message(chunk, cfg.meeting_context, part)
    raw = llm_generate(SYSTEM_PROMPT, user, cfg); obj = parse_json_obj(raw); attempts = 1
    if obj is None:
        raw = llm_generate(SYSTEM_PROMPT, user + "\n\nYour previous reply was not valid JSON. Reply with ONLY the JSON object.", cfg)
        obj = parse_json_obj(raw); attempts = 2
    warn = None
    if obj is None:
        obj, warn = dict(EMPTY), "LLM returned invalid JSON twice; chunk contributed nothing"
    return obj, raw, {"attempts": attempts, "warning": warn}

print("document_chunk ready")

## Cell 11 — Run Stage 3: map over chunks, verify, reduce

Single-chunk meetings take one call. Multi-chunk meetings: per-chunk extraction → verification → merge by
evidence lines → one short reduce call for the overall summary.

In [ ]:
def _dedupe(items: list[dict], key: str) -> list[dict]:
    seen, out = set(), []
    for it in items:
        k = (tuple(it["evidence_lines"]), _norm(it[key])[:60])
        if k not in seen:
            seen.add(k); out.append(it)
    return out

def run_stage3(segments: list[dict], cfg: DocConfig, on_progress=None) -> dict:
    t0 = time.time(); lines = {s["id"]: s["text"] for s in segments}
    chunks = make_chunks(segments, cfg.max_chunk_words, cfg.chunk_overlap_lines)
    parts, warnings, raw_outputs = [], [], []
    for ci, chunk in enumerate(chunks):
        part = f"(Part {ci+1} of {len(chunks)} of the meeting.)\n" if len(chunks) > 1 else ""
        obj, raw, info = document_chunk(chunk, cfg, part); raw_outputs.append(raw)
        if info["warning"]: warnings.append(f"chunk {ci+1}: {info['warning']}")
        parts.append(verify_output(obj, lines, cfg))
        if on_progress: on_progress("documenting", (ci + 1) / len(chunks))
    # ---- reduce ----
    merged = {
        "summary": parts[0]["summary"] if len(parts) == 1 else "",
        "discussion_points": [d for p in parts for d in p["discussion_points"]],
        "decisions": _dedupe([d for p in parts for d in p["decisions"]], "statement"),
        "proposals_not_adopted": _dedupe([d for p in parts for d in p["proposals_not_adopted"]], "statement"),
        "open_questions": _dedupe([d for p in parts for d in p["open_questions"]], "statement"),
        "action_items": _dedupe([a for p in parts for a in p["action_items"]], "task"),
        "verification": {"dropped": [x for p in parts for x in p["verification"]["dropped"]],
                         "adjusted": [x for p in parts for x in p["verification"]["adjusted"]]},
    }
    if len(parts) > 1:                       # one short call to write the overall summary from the part summaries
        joined = "\n".join(f"Part {i+1}: {p['summary']}" for i, p in enumerate(parts))
        raw = llm_generate("You write a 3-5 sentence meeting summary from part summaries. Use only what is given. Return plain text.",
                           joined, cfg)
        merged["summary"] = raw.strip()
    merged["warnings"] = warnings
    merged["meta"] = {"documenter": f"{cfg.model_id} (4-bit NF4)" if cfg.load_in_4bit else cfg.model_id,
                      "n_chunks": len(chunks), "settings": vars(cfg), "elapsed_s": round(time.time() - t0, 1),
                      "n_dropped": len(merged["verification"]["dropped"]), "n_adjusted": len(merged["verification"]["adjusted"])}
    merged["_raw_outputs"] = raw_outputs
    return merged

def show_progress(stage, p): print(f"  [{p*100:5.1f}%] {stage}")

R3 = run_stage3(SEGMENTS, CFG, on_progress=show_progress)
m = R3["meta"]
print(f"\n{m['documenter']} | {m['n_chunks']} chunk(s) in {m['elapsed_s']}s | decisions {len(R3['decisions'])} | "
      f"proposals {len(R3['proposals_not_adopted'])} | actions {len(R3['action_items'])} | "
      f"dropped by verifier {m['n_dropped']} | fields set to unspecified {m['n_adjusted']}")

## Cell 12 — Inspect the verified result

In [ ]:
print("SUMMARY:", R3["summary"], "\n")
for d in R3["discussion_points"]:
    print(f"• {d['topic']}"); [print("    -", p) for p in d["points"]]
print("\nDECISIONS:")
for d in R3["decisions"]:
    print(f"  ✔ {d['statement']}\n      evidence L{d['evidence_lines']}: \"{d['evidence_quote']}\"")
print("\nPROPOSALS NOT ADOPTED:")
for d in R3["proposals_not_adopted"]:
    print(f"  ✗ {d['statement']}  [{d['type']}]  L{d['evidence_lines']}")
print("\nOPEN QUESTIONS:")
for d in R3["open_questions"]:
    print(f"  ? {d['statement']}  L{d['evidence_lines']}")
print("\nACTION ITEMS:")
for a in R3["action_items"]:
    print(f"  □ {a['task']}  | owner: {a['owner']} | deadline: {a['deadline']} | L{a['evidence_lines']}")
print("\nVERIFIER DROPPED:")
for x in R3["verification"]["dropped"]:
    print(f"  - [{x['kind']}] {x['item'].get('statement') or x['item'].get('task')!r}: {x['reason']}")
print("\nVERIFIER ADJUSTED:")
for x in R3["verification"]["adjusted"]:
    print(f"  - {x['task']!r}: {x['note']}")
for w in R3["warnings"]:
    print("WARNING:", w)

## Cell 13 — Render Markdown from the verified JSON (by code, never by the LLM)

This guarantees the PS requirement that both formats carry the same decisions and tasks.

In [ ]:
def render_markdown(rec: dict) -> str:
    meta, mins = rec["meta"], rec["minutes"]
    L = [f"# Meeting record — {meta.get('audio_file', 'recording')}", "",
         f"*Duration:* {meta.get('duration_s', '?')} s · *STT:* {meta['models'].get('stt')} · "
         f"*Refiner:* {meta['models'].get('refiner')} · *Documenter:* {meta['models'].get('documenter')}", "",
         "## Summary", "", mins["summary"] or "_No summary produced._", "", "## Minutes", ""]
    for d in mins["discussion_points"]:
        L.append(f"**{d['topic']}**"); L += [f"- {p}" for p in d["points"]]; L.append("")
    if not mins["discussion_points"]: L += ["_No discussion points recorded._", ""]
    L += ["## Key decisions", ""]
    L += [f"{i+1}. {d['decision']}  \n   _Evidence (L{', L'.join(f'{x:03d}' for x in d['evidence_lines'])}):_ \"{d['evidence_quote']}\""
          for i, d in enumerate(rec["decisions"])] or ["_No decisions were reached in this meeting._"]
    L.append("")
    if rec["proposals_not_adopted"]:
        L += ["## Proposals discussed but not adopted", ""]
        L += [f"- {p['statement']} _(L{', L'.join(f'{x:03d}' for x in p['evidence_lines'])})_" for p in rec["proposals_not_adopted"]]
        L.append("")
    if rec.get("open_questions"):
        L += ["## Open questions", ""] + [f"- {q['statement']}" for q in rec["open_questions"]] + [""]
    L += ["## Action items", ""]
    if rec["action_items"]:
        L += ["| # | Task | Owner | Deadline | Evidence |", "|---|---|---|---|---|"]
        L += [f"| {i+1} | {a['task']} | {a['owner']} | {a['deadline']} | L{', L'.join(f'{x:03d}' for x in a['evidence_lines'])} |"
              for i, a in enumerate(rec["action_items"])]
    else:
        L.append("_No action items were assigned._")
    L += ["", "---", f"_Owners and deadlines are shown as **unspecified** when not stated in the recording. "
          f"Items dropped by verification: {rec['meta'].get('documenter_verification', {}).get('dropped', 0)}._"]
    return "\n".join(L)

print("render_markdown ready")

## Cell 14 — Assemble the final record and save

`meeting_record.json` is the Appendix B contract, fully filled. `meeting_record.md` is rendered from it.
`verification_log.json` keeps everything the verifier dropped or adjusted (tech-note evidence).

In [ ]:
from datetime import datetime
import copy

def assemble_record(s2: dict, r3: dict) -> dict:
    rec = copy.deepcopy(s2["record"]) if isinstance(s2["record"], dict) else {}
    rec.setdefault("meta", {}).setdefault("models", {})["documenter"] = r3["meta"]["documenter"]
    rec["meta"]["documenter_settings"] = r3["meta"]["settings"]
    rec["meta"]["documenter_warnings"] = r3["warnings"]
    rec["meta"]["documenter_verification"] = {"dropped": r3["meta"]["n_dropped"], "adjusted": r3["meta"]["n_adjusted"]}
    rec["minutes"] = {"summary": r3["summary"], "discussion_points": r3["discussion_points"]}
    rec["decisions"] = [{"decision": d["statement"], "evidence_quote": d["evidence_quote"], "evidence_lines": d["evidence_lines"]}
                        for d in r3["decisions"]]
    rec["proposals_not_adopted"] = [{"statement": p["statement"], "type": p["type"], "evidence_quote": p["evidence_quote"],
                                     "evidence_lines": p["evidence_lines"]} for p in r3["proposals_not_adopted"]]
    rec["open_questions"] = [{"statement": q["statement"], "evidence_lines": q["evidence_lines"]} for q in r3["open_questions"]]
    rec["action_items"] = [{"task": a["task"], "owner": a["owner"], "deadline": a["deadline"],
                            "owner_stated": a["owner_stated"], "deadline_stated": a["deadline_stated"],
                            "evidence_quote": a["evidence_quote"], "evidence_lines": a["evidence_lines"]} for a in r3["action_items"]]
    return rec

def save_stage3(s2: dict, r3: dict, out_dir: Path) -> tuple[Path, dict]:
    run_dir = out_dir / f"stage3_{datetime.now():%Y%m%d_%H%M%S}"; run_dir.mkdir(parents=True, exist_ok=True)
    rec = assemble_record(s2, r3)
    (run_dir / "meeting_record.json").write_text(json.dumps(rec, indent=2, ensure_ascii=False), encoding="utf-8")
    (run_dir / "meeting_record.md").write_text(render_markdown(rec), encoding="utf-8")
    (run_dir / "verification_log.json").write_text(json.dumps(
        {"dropped": r3["verification"]["dropped"], "adjusted": r3["verification"]["adjusted"], "warnings": r3["warnings"],
         "meta": r3["meta"], "raw_model_outputs": r3["_raw_outputs"]}, indent=2, ensure_ascii=False), encoding="utf-8")
    # the PS asks for raw + refined as separate downloads too; copy them beside the record when available
    for key, name in (("raw_transcript", "raw_transcript.txt"), ("refined_transcript", "refined_transcript.txt")):
        t = rec.get(key); t = t.get("text") if isinstance(t, dict) else t
        if isinstance(t, str) and t.strip(): (run_dir / name).write_text(t, encoding="utf-8")
    print("Saved to", run_dir)
    for f in sorted(run_dir.iterdir()): print(f"  {f.name:<26} {f.stat().st_size/1024:7.1f} KB")
    return run_dir, rec

RUN_DIR, RECORD = save_stage3(S2, R3, OUT_DIR)
print("\n" + render_markdown(RECORD))

In [ ]:
# Cell 14a: Calendar Generator (.ics)
import re
from datetime import datetime, timedelta
from pathlib import Path
from icalendar import Calendar, Event

def _parse_relative_deadline(deadline_str: str, base_time: datetime = None) -> datetime:
    base = base_time or datetime.now()
    d = (deadline_str or "").lower().strip()
    weekdays = {
        "monday": 0, "tuesday": 1, "wednesday": 2, "thursday": 3,
        "friday": 4, "saturday": 5, "sunday": 6,
        "mon": 0, "tue": 1, "wed": 2, "thu": 3, "fri": 4, "sat": 5, "sun": 6
    }
    for name, target_idx in weekdays.items():
        if re.search(r"\b" + name + r"\b", d):
            days_ahead = (target_idx - base.weekday() + 7) % 7
            days_ahead = 7 if days_ahead == 0 else days_ahead
            return (base + timedelta(days=days_ahead)).replace(hour=17, minute=0, second=0, microsecond=0)

    if "tomorrow" in d:
        return (base + timedelta(days=1)).replace(hour=17, minute=0, second=0, microsecond=0)
    if "next week" in d or "eow" in d:
        return (base + timedelta(days=7)).replace(hour=17, minute=0, second=0, microsecond=0)

    return (base + timedelta(days=3)).replace(hour=17, minute=0, second=0, microsecond=0)

def export_action_items_to_ics(record: dict, output_path: str | Path) -> Path:
    cal = Calendar()
    cal.add("prodid", "-//AI Meeting Assistant//EN")
    cal.add("version", "2.0")

    actions = record.get("action_items", [])
    for item in actions:
        task = item.get("task", "").strip()
        if not task:
            continue
        due_time = _parse_relative_deadline(item.get("deadline", "unspecified"))
        event = Event()
        event.add("summary", f"Task: {task}")
        event.add("dtstart", due_time)
        event.add("dtend", due_time + timedelta(hours=1))
        event.add("description", (
            f"Owner: {item.get('owner', 'unspecified')}\n"
            f"Deadline: {item.get('deadline', 'unspecified')}\n"
            f"Evidence (L{item.get('evidence_lines', [])}): \"{item.get('evidence_quote', '')}\""
        ))
        cal.add_component(event)

    out_p = Path(output_path)
    out_p.write_bytes(cal.to_ical())
    print(f"Generated calendar: {out_p.name} ({out_p.stat().st_size / 1024:.1f} KB)")
    return out_p

ICS_PATH = RUN_DIR / "meeting_action_items.ics"
export_action_items_to_ics(RECORD, ICS_PATH)

In [ ]:
# Cell 14b: Grounded Mini Chatbot
from rapidfuzz import fuzz

CHATBOT_SYSTEM = """You are a strictly grounded meeting query assistant.
Answer the user's question using ONLY the provided meeting lines.
- Always cite the line number(s) (e.g. [L003]).
- If the question was NOT answered or discussed in the meeting lines, reply EXACTLY:
  "This topic was not discussed or confirmed in the meeting."
- Do NOT guess, extrapolate, or introduce external knowledge."""

def ask_meeting(query: str, numbered_lines: list[str], max_lines: int = 6) -> str:
    # 1. Fuzzy Retrieval over transcript
    scored = []
    for line in numbered_lines:
        score = fuzz.token_set_ratio(query, line)
        scored.append((score, line))
    scored.sort(key=lambda x: x[0], reverse=True)
    context = "\n".join([line for _, line in scored[:max_lines]])

    user_msg = f"Meeting Lines:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    messages = [{"role": "system", "content": CHATBOT_SYSTEM}, {"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Test query with loaded Qwen2.5-7B
numbered_transcript = S2.get("record", {}).get("refined_transcript", {}).get("lines", []) or [
    f"[L{s['id']:03d}] {s['text']}" for s in SEGMENTS
]
print("Query 1:", ask_meeting("When is the launch release date?", numbered_transcript))
print("Query 2:", ask_meeting("What was decided about the logo color?", numbered_transcript))

## Cell 15 — Evaluation against gold labels (tech-note table)

Fill in what the recording really contains. Reports decision precision/recall (a proposal reported as a
decision is the key false positive), action-item precision/recall, invented owner/deadline count, evidence
verification rate and JSON validity.

In [ ]:
GOLD = {   # fill from your own clip; leave lists empty to skip. Match is fuzzy on the text.
    "decisions": [],          # e.g. ["release date is the twenty third of October", "will not migrate production this quarter"]
    "not_decisions": [],      # proposals that must NOT appear as decisions, e.g. ["API gateway to AWS"]
    "action_items": [],       # e.g. [{"task": "Terraform scripts", "owner": "Priya", "deadline": "by Friday"},
                              #       {"task": "rollback runbook", "owner": "unspecified", "deadline": "unspecified"},
                              #       {"task": "benchmark report", "owner": "Rahul", "deadline": "unspecified"}]
}

def _match(needle: str, hay: str) -> bool:
    return fuzz.partial_ratio(_norm(needle), _norm(hay)) >= 80

if any(GOLD.values()):
    dec_txt = [d["decision"] + " " + d["evidence_quote"] for d in RECORD["decisions"]]
    tp = sum(any(_match(g, d) for d in dec_txt) for g in GOLD["decisions"])
    fp_prop = sum(any(_match(g, d) for d in dec_txt) for g in GOLD["not_decisions"])
    print(f"decisions: recall {tp}/{len(GOLD['decisions'])} | precision {tp}/{len(dec_txt)} | proposals reported as decisions: {fp_prop} (target 0)")
    hit, inv_owner, inv_dead = 0, 0, 0
    for g in GOLD["action_items"]:
        found = [a for a in RECORD["action_items"] if _match(g["task"], a["task"] + " " + a["evidence_quote"])]
        if found:
            hit += 1; a = found[0]
            inv_owner += (g["owner"] == "unspecified" and a["owner"] != "unspecified")
            inv_dead  += (g["deadline"] == "unspecified" and a["deadline"] != "unspecified")
    print(f"action items: recall {hit}/{len(GOLD['action_items'])} | precision {hit}/{len(RECORD['action_items'])} | "
          f"invented owners {inv_owner} | invented deadlines {inv_dead} (targets 0)")
n_items = len(R3["decisions"]) + len(R3["proposals_not_adopted"]) + len(R3["open_questions"]) + len(R3["action_items"])
print(f"evidence verification: {n_items} kept, {R3['meta']['n_dropped']} dropped, {R3['meta']['n_adjusted']} fields set to unspecified")
print(f"JSON validity: {'OK' if not R3['warnings'] else 'FAILED on ' + str(len(R3['warnings'])) + ' chunk(s)'}")

## Cell 16 — Free the GPU

In [ ]:
def free_llm():
    global llm, tokenizer
    for name in ("llm", "tokenizer"):
        if name in globals(): del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); print(f"GPU mem now {torch.cuda.memory_allocated()/1e9:.1f} GB")

# free_llm()